In [8]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

✅ 로컬 Jupyter


In [9]:
# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

✅ API 키 OK


In [10]:
# 패키지 import

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

print("✅ import 완료")

✅ import 완료


In [11]:
import json
with open("personas.json", encoding="utf-8") as f:
    PERSONAS = json.load(f)

In [12]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다."),
    ("human", "{question}")
])   # LCEL 파이프 연산자

# 챗봇 핵심 함수: 페르소나 이름과 질문을 받아 답변 반환
def ask(persona_name: str, question: str) -> str:
    persona = PERSONAS[persona_name]

    system_message = persona["system"]
    temperature = persona["temperature"]

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=temperature)

    chain = prompt | llm

    # 2. 체인 호출 (2교시에서 배운 prompt | llm 패턴)
    result = chain.invoke({"role": system_message, "question": question})

    # 3. AIMessage 객체에서 답변 텍스트만 꺼내서 반환
    return result.content

In [18]:
# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /list, /help")

    elif cmd == "/list":
        print("🎭 사용 가능한 페르소나:")
        
        for name, info in PERSONAS.items():
            print(f"- {name}: {info['description']}")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

In [19]:
# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")

In [17]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

🎭 사용 가능한 페르소나:
- 분석가: 당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.
- 마케터: 당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.
- 개발자: 당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.
- 기획자: 당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.
- 디자이너: 당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.
👋 종료합니다.
